# 🏆 Kaggle Playground Series S6E7: Final Submission #1 (Private LB Honest Model)
# 75-Model GPU Triad Ensemble with Two-Stage Nelder-Mead Optimization

---

### 📌 Executive Summary & Architecture Overview
This notebook contains the complete, production-grade **Private Leaderboard Primary Track (Honest Model)** pipeline for the Kaggle Playground Series S6E7 (*Predicting Student Health Risk*).

#### 🎯 Strategic Objective:
To achieve maximum statistical generalization and guard against Private Leaderboard shake-down by enforcing strict **Out-Of-Fold (OOF) Stratified Cross-Validation**, **Leak-Free Feature Engineering**, **Multi-Model Triad Ensembling**, and **Metric-Aware Threshold Calibration**.

#### 🏗️ Key Architecture Components:
1. **Leak-Free Feature Engineering (58+ Features)**:
   - **Missingness Topology Flags**: Row-level missingness counts and indicator columns.
   - **Clinical & Domain Bins**: Standard BMI weight status categories and Heart Rate physiological zones.
   - **Lifestyle Composite Risk Index**: Multi-marker risk score aggregating sleep, stress, activity, smoking, and BMI.
   - **Domain Ratios & Interactions**: Physical strain indicators (calories/step, water/BMI, sleep/exercise ratio).
   - **Leak-Free Peer Aggregations**: Target group statistics calculated purely on training folds.
   - **Integer Frequency Encodings**: Value counts computed exclusively from training data.
2. **Raw OOF Target Encoding (3 High-Cardinality Combos)**:
   - Out-of-fold target encoding for `stress_activity_combo`, `sleep_stress_combo`, and `lifestyle_triad` to prevent target leakage.
3. **Multi-Seed 5-Fold Triad Ensemble (75 Models Total)**:
   - **5 Seeds × 5 Folds × 3 GBDT Families** = **75 Models** (LightGBM + XGBoost GPU + CatBoost GPU).
4. **Two-Stage Scipy Nelder-Mead Optimization**:
   - **Stage 1**: Optimizes model blending weights ($w_{LGB}, w_{XGB}, w_{CAT}$) for maximum Out-Of-Fold Balanced Accuracy.
   - **Stage 2**: Optimizes class-wise decision boundary multipliers to maximize equal recall across all minority health condition classes (`fit`, `at-risk`, `unhealthy`).

---


## 1. Environment Setup & Dependency Imports
Initialize core Data Science, Machine Learning, and Optimization libraries. Disable non-critical warnings to ensure clean log outputs.

In [ ]:
# -------- CELL 1: INITIALIZATION & IMPORTS --------
# Import os module to handle operating system and directory path operations
import os
# Import gc module to perform manual garbage collection for memory optimization
import gc
# Import warnings module to control and suppress warning messages during execution
import warnings
# Import numpy for high-performance vectorized numerical and matrix operations
import numpy as np
# Import pandas for tabular data manipulation, DataFrame handling, and CSV I/O
import pandas as pd
# Import StratifiedKFold for class-balanced cross-validation splitting
from sklearn.model_selection import StratifiedKFold
# Import evaluation metrics to assess model performance and print diagnostic reports
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
# Import LabelEncoder to convert textual target categories into numeric zero-indexed labels
from sklearn.preprocessing import LabelEncoder
# Import minimize function from scipy optimization package for Nelder-Mead hyper-calibration
from scipy.optimize import minimize
# Import LightGBM gradient boosting framework
import lightgbm as lgb
# Import XGBoost gradient boosting framework
import xgboost as xgb
# Import CatBoostClassifier for handling categorical gradient boosting
from catboost import CatBoostClassifier

# Suppress non-critical Python and library warnings for clean terminal and cell outputs
warnings.filterwarnings('ignore')
# Display initialization status log to confirm environment is ready
print("Environment initialized successfully for Grandmaster v3 Master Pipeline.")


## 2. Data Ingestion & Path Configuration
Load raw training (`train.csv`) and testing (`test.csv`) datasets. Auto-detect dataset directory paths and configure target and ID variables.

In [ ]:
# -------- CELL 2: DATA LOADING & PATH SETUP --------
# Define default relative directory path where Kaggle raw datasets are stored
DATA_PATH = './data'
# Check if relative data directory exists; if not, set fallback absolute path to local workspace folder
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'c:/Users/mihisara/Desktop/ML/data'

# Read Kaggle training dataset CSV file into pandas DataFrame
train = pd.read_csv(f'{DATA_PATH}/train.csv')
# Read Kaggle test dataset CSV file into pandas DataFrame
test = pd.read_csv(f'{DATA_PATH}/test.csv')
# Define expected file path for Kaggle sample submission file
sub_file = f'{DATA_PATH}/sample_submission.csv'
# Check if sample submission file exists on disk
if os.path.exists(sub_file):
    # Read sample submission file into pandas DataFrame
    sample_submission = pd.read_csv(sub_file)
else:
    # Create dummy sample submission DataFrame using test IDs if file is missing
    sample_submission = pd.DataFrame({'id': test['id'], 'health_condition': 'at-risk'})

# Set target variable column name
TARGET = 'health_condition'
# Fallback: If target name is not in train columns, auto-detect column unique to train
if TARGET not in train.columns:
    TARGET = [c for c in train.columns if c not in test.columns][0]
# Retrieve ID column name from sample submission DataFrame header
ID_COLUMN = sample_submission.columns[0]

# Initialize list of columns to drop from feature set (starts with target column)
drop_columns = [TARGET]
# If ID column exists in training DataFrame, add it to drop columns list
if ID_COLUMN in train.columns: drop_columns.append(ID_COLUMN)

# Create feature set X by dropping target and ID columns from train DataFrame
X = train.drop(columns=drop_columns).copy()
# Create ground truth target series y_text containing raw textual labels
y_text = train[TARGET].copy()
# Create test feature set by dropping ID column from test DataFrame safely
test_features = test.drop(columns=[ID_COLUMN], errors='ignore').copy()
# Output DataFrame dimensions to verify data loading integrity
print(f'Train shape: {X.shape}, Test shape: {test_features.shape}')


## 3. Leak-Free Master Feature Engineering (v3)
Generate domain-specific features without data leakage between train and test datasets:
1. **Missingness Flags**: Captures row-level missing values.
2. **Medical & Domain Bins**: Categorizes BMI into WHO standard weight categories and Heart Rate into clinical zones.
3. **Lifestyle Composite Risk Index**: Calculates a 0–6 composite risk score based on lifestyle thresholds.
4. **Domain Ratios & Interactions**: Physical strain and efficiency metrics.
5. **Leak-Free Peer Group Aggregations**: Group statistics calculated purely on train set and mapped to test.
6. **Integer Frequency Encodings**: Value counts calculated strictly from train set.

In [ ]:
# -------- CELL 3: LEAK-FREE FEATURE ENGINEERING --------
# Define master feature engineering function receiving train and test feature DataFrames
def add_v3_master_features(df_tr, df_te):
    # Create defensive copy of training DataFrame to prevent mutating original data
    df_tr = df_tr.copy()
    # Create defensive copy of test DataFrame to prevent mutating original data
    df_te = df_te.copy()

    # 1. Missingness-Aware Flags
    # Get list of original column names prior to feature generation
    orig_cols = df_tr.columns.tolist()
    # Calculate total missing value count per row for training dataset
    df_tr['row_missing_count'] = df_tr[orig_cols].isnull().sum(axis=1).astype('int8')
    # Calculate total missing value count per row for test dataset
    df_te['row_missing_count'] = df_te[orig_cols].isnull().sum(axis=1).astype('int8')
    # Loop through each original column to create binary missingness indicator flags
    for col in orig_cols:
        # Add binary missing flag column for train set (1 if missing, 0 otherwise)
        df_tr[f'{col}__missing'] = df_tr[col].isna().astype('int8')
        # Add binary missing flag column for test set (1 if missing, 0 otherwise)
        df_te[f'{col}__missing'] = df_te[col].isna().astype('int8')

    # Create lowercase mapping dictionary to find column names regardless of casing
    lower_map = {c.lower(): c for c in df_tr.columns}
    # Helper function to match possible column aliases to actual DataFrame column names
    def find_col(possible_names):
        # Iterate over candidate column names
        for name in possible_names:
            # Return matching column name if present in mapping dictionary
            if name in lower_map: return lower_map[name]
        # Return None if no matching column alias is found
        return None

    # Resolve canonical column names for health markers
    sleep_col = find_col(['sleep_duration', 'sleep_hours', 'sleep'])
    exercise_col = find_col(['exercise_duration'])
    stress_col = find_col(['stress_level'])
    activity_col = find_col(['physical_activity_level', 'physical_activity'])
    quality_col = find_col(['sleep_quality'])
    smoke_col = find_col(['smoking_alcohol'])
    step_col = find_col(['step_count'])
    water_col = find_col(['water_intake'])
    hr_col = find_col(['heart_rate'])
    bmi_col = find_col(['bmi'])
    cal_col = find_col(['calorie_expenditure'])
    gender_col = find_col(['gender'])

    # 2. Medical & Domain Bins
    # Iterate through both train and test DataFrames to apply clinical binning
    for df in [df_tr, df_te]:
        # Bin BMI values into standard medical body mass categories
        if bmi_col:
            # Extract BMI column values
            bv = df[bmi_col]
            # Initialize zero integer array for BMI categories
            bcat = np.zeros(len(df), dtype='int8')
            # Category 1: Underweight (< 18.5)
            bcat[bv < 18.5] = 1
            # Category 2: Normal Weight (18.5 - 24.9)
            bcat[(bv >= 18.5) & (bv < 25.0)] = 2
            # Category 3: Overweight (25.0 - 29.9)
            bcat[(bv >= 25.0) & (bv < 30.0)] = 3
            # Category 4: Obese (>= 30.0)
            bcat[bv >= 30.0] = 4
            # Assign generated BMI categories to DataFrame
            df['bmi_category'] = bcat

        # Bin Heart Rate values into physiological response zones
        if hr_col:
            # Extract Heart Rate column values
            hv = df[hr_col]
            # Initialize zero integer array for Heart Rate zones
            hz = np.zeros(len(df), dtype='int8')
            # Zone 1: Bradycardia / Low Heart Rate (< 60 bpm)
            hz[hv < 60.0] = 1
            # Zone 2: Normal Resting Heart Rate (60 - 100 bpm)
            hz[(hv >= 60.0) & (hv <= 100.0)] = 2
            # Zone 3: Tachycardia / Elevated Heart Rate (> 100 bpm)
            hz[hv > 100.0] = 3
            # Assign generated Heart Rate zones to DataFrame
            df['hr_zone'] = hz

        # 3. Lifestyle Composite Risk Index (0 to 6 Score)
        # Initialize zero integer array for cumulative lifestyle risk score
        risk = np.zeros(len(df), dtype='int8')
        # Add +1 risk point if sleep duration is sub-optimal (< 6.0 hours)
        if sleep_col: risk += (df[sleep_col] < 6.0).astype('int8')
        # Add +1 risk point if psychological stress level is high
        if stress_col: risk += (df[stress_col].astype(str) == 'high').astype('int8')
        # Add +1 risk point if sleep quality is self-reported as poor
        if quality_col: risk += (df[quality_col].astype(str) == 'poor').astype('int8')
        # Add +1 risk point if physical activity level is sedentary
        if activity_col: risk += (df[activity_col].astype(str) == 'sedentary').astype('int8')
        # Add +1 risk point if smoking/alcohol usage is present
        if smoke_col: risk += (df[smoke_col].astype(str) == 'yes').astype('int8')
        # Add +1 risk point if BMI is elevated (>= 25.0)
        if bmi_col: risk += (df[bmi_col] >= 25.0).astype('int8')
        # Assign composite risk index column to DataFrame
        df['lifestyle_risk_index'] = risk

        # 4. Domain Ratios & Interactions
        if sleep_col:
            # Compute absolute deviation from ideal 8-hour sleep duration
            df['sleep_distance_from_8'] = (df[sleep_col] - 8.0).abs()
            # Create binary flag for healthy sleep duration (>= 7 hours)
            df['sleep_ge_7'] = (df[sleep_col] >= 7.0).astype(int)
            # Create binary flag for sleep deprivation (< 6 hours)
            df['sleep_lt_6'] = (df[sleep_col] < 6.0).astype(int)

        # Calculate metabolic efficiency ratio: Calories expended per step taken
        if cal_col and step_col: df['calories_per_step'] = df[cal_col] / (df[step_col] + 1.0)
        # Calculate caloric intensity ratio: Calories expended per exercise minute
        if cal_col and exercise_col: df['calories_per_exercise_min'] = df[cal_col] / (df[exercise_col] + 1.0)
        # Calculate energy-to-mass ratio: Calories expended relative to BMI
        if cal_col and bmi_col: df['calories_per_bmi'] = df[cal_col] / (df[bmi_col] + 0.01)
        # Calculate hydration ratio: Water intake relative to body mass index
        if water_col and bmi_col: df['water_per_bmi'] = df[water_col] / (df[bmi_col] + 0.01)
        # Calculate activity density ratio: Daily steps relative to BMI
        if step_col and bmi_col: df['step_per_bmi'] = df[step_col] / (df[bmi_col] + 0.01)
        # Calculate cardiac stress ratio: Heart rate per sleep hour
        if hr_col and sleep_col: df['hr_per_sleep'] = df[hr_col] / (df[sleep_col] + 0.01)
        # Calculate exertion heart rate ratio: Heart rate per exercise minute
        if hr_col and exercise_col: df['hr_per_exercise'] = df[hr_col] / (df[exercise_col] + 1.0)
        if sleep_col and exercise_col:
            # Compute multiplicative interaction between sleep duration and exercise duration
            df['sleep_exercise_interaction'] = df[sleep_col] * df[exercise_col]
            # Compute ratio between sleep duration and exercise duration
            df['sleep_exercise_ratio'] = df[sleep_col] / (df[exercise_col] + 0.01)
        # Calculate hydration activity ratio: Water intake per 1,000 steps
        if water_col and step_col: df['water_per_step'] = df[water_col] / (df[step_col] + 1.0)
        # Calculate exercise efficiency ratio: Exercise duration per step
        if exercise_col and step_col: df['exercise_per_step'] = df[exercise_col] / (df[step_col] + 1.0)

        # Categorical Combinations
        # Combine stress level and physical activity level into string composite feature
        if stress_col and activity_col:
            df['stress_activity_combo'] = df[stress_col].astype(str) + "_" + df[activity_col].astype(str)
        # Combine sleep quality and stress level into string composite feature
        if quality_col and stress_col:
            df['sleep_stress_combo'] = df[quality_col].astype(str) + "_" + df[stress_col].astype(str)
        # Combine stress level, activity level, and sleep quality into 3-way lifestyle triad feature
        if stress_col and activity_col and quality_col:
            df['lifestyle_triad'] = df[stress_col].astype(str) + "_" + df[activity_col].astype(str) + "_" + df[quality_col].astype(str)

    # 5. LEAK-FREE Peer Group Aggregations (Calculated strictly on train and mapped to test)
    if sleep_col and activity_col:
        # Calculate mean sleep duration per activity group on training set only
        grp = df_tr.groupby(activity_col)[sleep_col].mean().to_dict()
        # Compute sleep deviation from peer group mean for train set
        df_tr['sleep_diff_from_activity_mean'] = df_tr[sleep_col] - df_tr[activity_col].map(grp)
        # Compute sleep deviation from peer group mean for test set using train dictionary
        df_te['sleep_diff_from_activity_mean'] = df_te[sleep_col] - df_te[activity_col].map(grp)

    if exercise_col and stress_col:
        # Calculate mean exercise duration per stress level on training set only
        grp = df_tr.groupby(stress_col)[exercise_col].mean().to_dict()
        # Compute exercise deviation from stress peer group mean for train set
        df_tr['exercise_diff_from_stress_mean'] = df_tr[exercise_col] - df_tr[stress_col].map(grp)
        # Compute exercise deviation from stress peer group mean for test set using train dictionary
        df_te['exercise_diff_from_stress_mean'] = df_te[exercise_col] - df_te[stress_col].map(grp)

    if cal_col and activity_col:
        # Calculate mean calorie expenditure per activity level on training set only
        grp = df_tr.groupby(activity_col)[cal_col].mean().to_dict()
        # Compute calorie deviation from activity peer group mean for train set
        df_tr['calories_diff_from_activity_mean'] = df_tr[cal_col] - df_tr[activity_col].map(grp)
        # Compute calorie deviation from activity peer group mean for test set using train dictionary
        df_te['calories_diff_from_activity_mean'] = df_te[cal_col] - df_te[activity_col].map(grp)

    # 6. LEAK-FREE Integer Frequency Encoding (Value counts computed strictly from train set)
    # Define categorical columns to compute frequency count encodings
    cat_for_freq = [stress_col, activity_col, quality_col, smoke_col, gender_col, 'stress_activity_combo', 'sleep_stress_combo', 'lifestyle_triad']
    # Iterate through categorical columns list
    for c in cat_for_freq:
        # Check if column exists in training DataFrame
        if c and c in df_tr.columns:
            # Compute exact value counts dictionary from training dataset
            freq = df_tr[c].value_counts(dropna=False).to_dict()
            # Map frequency counts to training DataFrame and fill missing with 0
            df_tr[f'{c}__freq'] = df_tr[c].map(freq).fillna(0).astype('int32')
            # Map train frequency counts to test DataFrame and fill missing with 0
            df_te[f'{c}__freq'] = df_te[c].map(freq).fillna(0).astype('int32')

    # Return modified train and test DataFrames
    return df_tr, df_te

# Execute master feature engineering pipeline on X and test_features
X, test_features = add_v3_master_features(X, test_features)
# Print updated training DataFrame dimensions after feature engineering
print(f'Train shape after Leak-Free Feature Engineering: {X.shape}')


## 4. Out-Of-Fold (OOF) Target Encoding
Compute leak-free out-of-fold target encodings for high-cardinality composite categorical features (`stress_activity_combo`, `sleep_stress_combo`, `lifestyle_triad`) using 5-Fold Stratified Cross-Validation.

In [ ]:
# -------- CELL 4: RAW OOF TARGET ENCODING --------
# Initialize LabelEncoder instance to encode textual target classes into integers
le = LabelEncoder()
# Fit label encoder on target text series and transform to numeric integer array y
y = le.fit_transform(y_text)
# Extract total number of target classes (3 classes: at-risk, unhealthy, fit)
num_classes = len(le.classes_)

# List composite categorical features selected for out-of-fold target encoding
te_cols = ['stress_activity_combo', 'sleep_stress_combo', 'lifestyle_triad']
# Initialize 5-fold stratified cross-validation splitter for leak-free target encoding
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Log message indicating start of target encoding computation
print("Computing Raw OOF Target Encodings (Exact v3 Method)...")
# Loop over each targeted composite feature column
for c in te_cols:
    # Check if feature column exists in training DataFrame X
    if c in X.columns:
        # Loop through each individual target class index (0, 1, 2)
        for cls in range(num_classes):
            # Define new column name for class-specific target encoding
            col_name = f'{c}__te_cls{cls}'
            # Initialize target encoding column in train set with 0.0
            X[col_name] = 0.0
            # Initialize target encoding column in test set with 0.0
            test_features[col_name] = 0.0

            # Create binary target indicator array for current target class
            y_cls = (y == cls).astype(float)
            # Perform 5-fold split to compute out-of-fold target means for training set
            for tr_idx, va_idx in skf_te.split(X, y):
                # Compute target mean grouped by categorical value on training fold only
                grp_means = X.iloc[tr_idx].groupby(c).apply(lambda d: y_cls[d.index].mean()).to_dict()
                # Calculate global target mean across training fold as fallback
                global_mean = y_cls[tr_idx].mean()
                # Map group target means onto validation fold to prevent target leakage
                X.iloc[va_idx, X.columns.get_loc(col_name)] = X.iloc[va_idx][c].map(grp_means).fillna(global_mean)

            # Compute full target means across entire training dataset for test set mapping
            full_grp_means = X.groupby(c).apply(lambda d: y_cls[d.index].mean()).to_dict()
            # Calculate overall global target mean across entire training set
            global_mean = y_cls.mean()
            # Map target means to test set feature DataFrame
            test_features[col_name] = test_features[c].map(full_grp_means).fillna(global_mean)

# Identify all remaining string/categorical columns for data type formatting
categorical_columns = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
# Format categorical columns as pandas category dtypes for tree model compatibility
for col in categorical_columns:
    # Fill missing or null string values with 'missing' and convert to category dtype for train
    X[col] = X[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing').astype('category')
    # Fill missing or null string values with 'missing' and convert to category dtype for test
    test_features[col] = test_features[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing').astype('category')

# Print final training dataset shape after adding target encodings
print(f'Final Train shape after Raw OOF Target Encoding: {X.shape}')


## 5. Multi-Seed 5-Fold Triad Ensemble Engine (75 Models Total)
Train an ensemble across **5 Random Seeds × 5 Cross-Validation Folds × 3 GBDT Model Architectures = 75 Models**:
1. **LightGBM**: Fast CPU leaf-wise histogram tree growth.
2. **XGBoost**: CUDA GPU depth-wise tree growth with exact regularization.
3. **CatBoost**: CUDA GPU ordered categorical target statistics.

In [ ]:
# -------- CELL 5: MULTI-SEED 5-FOLD TRIAD ENSEMBLE --------
# Define list of 5 diverse random seeds for multi-seed variance reduction
SEEDS = [42, 2026, 999, 777, 3407]
# Initialize zero matrix to accumulate LightGBM out-of-fold probability predictions
oof_lgb = np.zeros((len(X), num_classes))
# Initialize zero matrix to accumulate XGBoost out-of-fold probability predictions
oof_xgb = np.zeros((len(X), num_classes))
# Initialize zero matrix to accumulate CatBoost out-of-fold probability predictions
oof_cat = np.zeros((len(X), num_classes))

# Initialize zero matrix to accumulate LightGBM test set probability predictions
test_lgb = np.zeros((len(test_features), num_classes))
# Initialize zero matrix to accumulate XGBoost test set probability predictions
test_xgb = np.zeros((len(test_features), num_classes))
# Initialize zero matrix to accumulate CatBoost test set probability predictions
test_cat = np.zeros((len(test_features), num_classes))

# Define LightGBM hyperparameter dictionary for multi-threaded CPU execution
lgb_params = {'n_estimators': 850, 'learning_rate': 0.03, 'num_leaves': 63, 'max_depth': 7, 'min_child_samples': 45, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'multiclass', 'num_class': num_classes, 'n_jobs': -1, 'verbose': -1}
# Define XGBoost hyperparameter dictionary configured for CUDA GPU acceleration
xgb_params = {'n_estimators': 850, 'learning_rate': 0.03, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'multi:softprob', 'num_class': num_classes, 'eval_metric': 'mlogloss', 'enable_categorical': True, 'n_jobs': -1, 'tree_method': 'hist', 'device': 'cuda', 'early_stopping_rounds': 50}
# Define CatBoost hyperparameter dictionary configured for CUDA GPU acceleration
cat_params = {'iterations': 850, 'learning_rate': 0.03, 'depth': 6, 'loss_function': 'MultiClass', 'task_type': 'GPU', 'verbose': 0}

# Get list of categorical column names for CatBoost explicitly
cat_cols = X.select_dtypes(include=['category', 'object']).columns.tolist()
# Create copy of X formatted with clean strings for CatBoost categorical feature support
X_cat = X.copy()
# Create copy of test_features formatted with clean strings for CatBoost
test_features_cat = test_features.copy()
# Replace missing string representations in categorical columns for CatBoost
for col in cat_cols:
    X_cat[col] = X_cat[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing')
    test_features_cat[col] = test_features_cat[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing')

# Log message signaling start of 75-model training loop
print(f'Starting 5-Seed 5-Fold Triad Ensemble ({len(SEEDS)*5*3} Models Total)...')
# Iterate through each specified random seed
for seed in SEEDS:
    # Print header for current seed training run
    print(f'\n--- Training Seed {seed} ---')
    # Instantiate StratifiedKFold cross-validator with current seed
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    # Iterate through each of the 5 cross-validation folds
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        # Slice feature matrix X into training and validation sets for current fold
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        # Slice target array y into training and validation labels for current fold
        y_tr, y_va = y[train_idx], y[valid_idx]
        # Slice CatBoost feature matrix X_cat into train and validation sets
        X_tr_cat, X_va_cat = X_cat.iloc[train_idx], X_cat.iloc[valid_idx]

        # --- 1. LightGBM Classifier Model Training ---
        p = lgb_params.copy(); p['random_state'] = seed
        # Instantiate LightGBM model with seed parameters
        m1 = lgb.LGBMClassifier(**p)
        # Fit LightGBM model on training fold with early stopping on validation fold
        m1.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
        # Accumulate out-of-fold probability predictions normalized by number of seeds
        oof_lgb[valid_idx] += m1.predict_proba(X_va) / len(SEEDS)
        # Accumulate test set probability predictions normalized by total folds and seeds
        test_lgb += m1.predict_proba(test_features) / (5.0 * len(SEEDS))

        # --- 2. XGBoost Classifier Model Training ---
        p = xgb_params.copy(); p['random_state'] = seed
        # Instantiate XGBoost model with seed parameters
        m2 = xgb.XGBClassifier(**p)
        # Fit XGBoost model on training fold with evaluation set
        m2.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        # Accumulate out-of-fold probability predictions normalized by number of seeds
        oof_xgb[valid_idx] += m2.predict_proba(X_va) / len(SEEDS)
        # Accumulate test set probability predictions normalized by total folds and seeds
        test_xgb += m2.predict_proba(test_features) / (5.0 * len(SEEDS))

        # --- 3. CatBoost Classifier Model Training ---
        p = cat_params.copy(); p['random_seed'] = seed
        # Instantiate CatBoost model with seed parameters
        m3 = CatBoostClassifier(**p)
        # Fit CatBoost model on training fold specifying categorical feature list
        m3.fit(X_tr_cat, y_tr, cat_features=cat_cols, eval_set=(X_va_cat, y_va), early_stopping_rounds=50, verbose=False)
        # Accumulate out-of-fold probability predictions normalized by number of seeds
        oof_cat[valid_idx] += m3.predict_proba(X_va_cat) / len(SEEDS)
        # Accumulate test set probability predictions normalized by total folds and seeds
        test_cat += m3.predict_proba(test_features_cat) / (5.0 * len(SEEDS))

# Print raw out-of-fold Balanced Accuracy score for LightGBM ensemble
print(f'\n5-Seed LightGBM Raw OOF Score: {balanced_accuracy_score(y, np.argmax(oof_lgb, axis=1)):.5f}')
# Print raw out-of-fold Balanced Accuracy score for XGBoost ensemble
print(f'5-Seed XGBoost Raw OOF Score:  {balanced_accuracy_score(y, np.argmax(oof_xgb, axis=1)):.5f}')
# Print raw out-of-fold Balanced Accuracy score for CatBoost ensemble
print(f'5-Seed CatBoost Raw OOF Score: {balanced_accuracy_score(y, np.argmax(oof_cat, axis=1)):.5f}')


## 6. Two-Stage Scipy Optimization & Error Analysis
Perform post-processing calibration using Scipy Nelder-Mead optimization:
- **Stage 1**: Find optimal model weights $[w_{LGB}, w_{XGB}, w_{CAT}]$ to blend probability predictions.
- **Stage 2**: Find optimal class multipliers $[c_0, c_1, c_2]$ to shift decision boundaries and equalize recall across minority classes (`fit`, `at-risk`, `unhealthy`).

In [ ]:
# -------- CELL 6: TWO-STAGE OPTIMIZATION & DIAGNOSTICS --------
# Log header indicating Stage 1 optimization start
print('\nStage 1: Optimizing Model Blending Weights (LGBM, XGB, CatBoost)...')
# Objective function for Stage 1: Returns negative Balanced Accuracy to minimize
def model_blend_objective(weights):
    # Unpack candidate model weights for LightGBM, XGBoost, and CatBoost
    w1, w2, w3 = weights
    # Compute weighted sum of out-of-fold probability predictions
    blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
    # Return negative balanced accuracy score (minimizing negative = maximizing accuracy)
    return -balanced_accuracy_score(y, np.argmax(blend, axis=1))

# Optimize model weights using Nelder-Mead simplex algorithm starting from [0.15, 0.15, 0.70]
res_m = minimize(model_blend_objective, [0.15, 0.15, 0.70], method='Nelder-Mead', bounds=[(0.0, 3.0)]*3)
# Extract optimal model weights from optimization result
w1, w2, w3 = res_m.x
# Output optimized model weights for LGBM, XGBoost, and CatBoost
print(f'Optimized Model Blending Weights (LGB, XGB, Cat): [{w1:.4f}, {w2:.4f}, {w3:.4f}]')

# Compute final blended out-of-fold probabilities using optimized weights
blended_oof = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
# Compute final blended test set probabilities using optimized weights
blended_test = w1 * test_lgb + w2 * test_xgb + w3 * test_cat

# Calculate raw blended out-of-fold Balanced Accuracy score before class calibration
raw_blend_cv = balanced_accuracy_score(y, np.argmax(blended_oof, axis=1))
# Output raw blended CV accuracy
print(f'Raw Blended Triad OOF Balanced Accuracy: {raw_blend_cv:.5f}')

# Log header indicating Stage 2 optimization start
print('\nStage 2: Optimizing Class-Wise Multipliers on Blended OOF...')
# Objective function for Stage 2: Optimizes class probability multipliers
def class_weight_objective(weights):
    # Scale blended out-of-fold probabilities by class-wise weight multipliers
    adj = blended_oof * weights
    # Return negative balanced accuracy score to minimize
    return -balanced_accuracy_score(y, np.argmax(adj, axis=1))

# Optimize class multipliers using Nelder-Mead starting from unit vector [1.0, 1.0, 1.0]
res_c = minimize(class_weight_objective, np.ones(num_classes), method='Nelder-Mead', bounds=[(0.1, 2.0)] * num_classes)
# Extract optimal class multiplier weights from optimization result
class_weights = res_c.x
# Output optimized class multipliers
print(f'Optimized Class Weights: {class_weights}')

# Apply optimized class multipliers to blended out-of-fold probabilities
final_oof_probs = blended_oof * class_weights
# Get final predicted class indices via argmax on calibrated probabilities
opt_preds = np.argmax(final_oof_probs, axis=1)
# Calculate final grandmaster OOF Balanced Accuracy score
master_cv_score = balanced_accuracy_score(y, opt_preds)
# Print final master CV score
print(f'\n🏆 FINAL GRANDMASTER V3 MASTER OOF BALANCED ACCURACY (TARGET: 0.95260+): {master_cv_score:.5f}')

# --- Diagnostic Error Analysis Report ---
print('\n--- Diagnostic Error Analysis ---')
# Generate confusion matrix comparing ground truth y with optimized predictions
cm = confusion_matrix(y, opt_preds)
# Print confusion matrix to terminal
print("Confusion Matrix:")
print(cm)

# Print detailed classification report showing precision, recall, and f1-score per class
print("\nClassification Report:")
print(classification_report(y, opt_preds, target_names=le.classes_))

# Calculate per-class recall scores (diagonal elements divided by row sums)
class_recalls = cm.diagonal() / cm.sum(axis=1)
# Print recall for each individual target class
for idx, class_name in enumerate(le.classes_):
    print(f"Recall for class '{class_name}': {class_recalls[idx]:.5f}")


## 7. Final Test Predictions & Submission Export
Apply optimized class multipliers to test set blended probabilities, map integer class indices back to original string labels (`fit`, `at-risk`, `unhealthy`), and export the final submission CSV file.

In [ ]:
# -------- CELL 7: FINAL SUBMISSION EXPORT --------
# Scale blended test set probabilities using optimized class multiplier weights
final_test_probs = blended_test * class_weights
# Select class index with maximum calibrated probability for each test sample
final_preds = np.argmax(final_test_probs, axis=1)
# Convert integer predicted class indices back to original textual health labels
final_labels = le.inverse_transform(final_preds)

# Create defensive copy of sample submission template DataFrame
submission = sample_submission.copy()
# Assign predicted textual health condition labels to target column
submission[TARGET] = final_labels
# Export final submission DataFrame to CSV format without index column
submission.to_csv('submission_grandmaster_v3_master_winner.csv', index=False)
# Output confirmation message stating submission CSV file was saved successfully
print('Saved submission_grandmaster_v3_master_winner.csv successfully!')

# Display first 5 rows of final submission DataFrame for visual verification
submission.head()
